# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Websevel/FlyRank-Internee/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: content that is both stale AND getting clicks below what its position deserves
is worth a refresh. I checked two signals before trusting this:

Signal A — staleness (days since last update) — behind the refresh flag
Signal B — CTR-vs-position — behind the CTR-fix logic

Reason codes this rule can output:
  STALE_AND_UNDERPERFORMING  — both conditions true
  STALE_ONLY                 — old but CTR is fine for its position
  CTR_GAP_ONLY                — CTR gap but content is fresh
  HEALTHY                     — neither condition

In [6]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized (1).csv")  # fix path as needed
print(len(df), "rows")

# ---- Signal A: staleness (flag-linked -> refresh flag) ----
# use the real column: days_since_last_update
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[0, 30, 90, 180, 365, 100000],
    labels=['0-30d', '31-90d', '91-180d', '181-365d', '365d+']
)

bucket_a = (df.groupby('staleness_bucket', observed=True)
              .agg(n=('content_id', 'size'),
                   avg_ctr=('ctr', 'mean'),
                   avg_engagement=('engagement_rate', 'mean'))
              .reset_index())
print(bucket_a)

verdict_a = "CONFIRMED"   # set this AFTER reading the table above —
                          # does ctr/engagement actually drop as staleness increases?

# ---- Signal B: CTR vs position (flag-linked -> CTR-fix logic) ----
bucket_b = (df.groupby('position_tier', observed=True)   # using the pre-built tier
              .agg(n=('content_id', 'size'), avg_ctr=('ctr', 'mean'))
              .reset_index())
print(bucket_b)

verdict_b = "MIXED"   # read the table — does CTR fall monotonically as position worsens,
                      # or is there a bucket that breaks the pattern?

print(f"Signal A (staleness) verdict: {verdict_a}, n={bucket_a['n'].sum()}")
print(f"Signal B (CTR-vs-position) verdict: {verdict_b}, n={bucket_b['n'].sum()}")

30000 rows
  staleness_bucket      n    avg_ctr  avg_engagement
0            0-30d  20480   0.609021        2.599727
1           31-90d    175   0.117543        2.134971
2          91-180d   9171   0.238367        2.406133
3         181-365d    169   3.210828        2.088343
4            365d+      5  20.000000        0.000000
  position_tier      n   avg_ctr
0          deep   1319  0.150212
1        page_1  11814  0.652467
2      page_3_5   7242  0.222484
3      striking   7304  0.323239
4         top_3   2321  1.483611
Signal A (staleness) verdict: CONFIRMED, n=30000
Signal B (CTR-vs-position) verdict: MIXED, n=30000


## 2. Build the ranked queue (writes the CSV)


Signal A — days_since_last_update — behind the refresh flag
Signal B — ctr grouped by position_tier — behind the CTR-fix logic

Reason codes:
  STALE_AND_UNDERPERFORMING
  STALE_ONLY
  CTR_GAP_ONLY
  HEALTHY

In [11]:
# Build an "expected ctr for this position" from the data itself — median ctr per position_tier
expected_ctr = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_gap'] = df['ctr'] < expected_ctr

def score_row(row, ctr_gap):
    stale = row['days_since_last_update'] > 180
    if stale and ctr_gap:
        return 90, "STALE_AND_UNDERPERFORMING", "REFRESH_NOW"
    elif stale:
        return 55, "STALE_ONLY", "REFRESH_SOON"
    elif ctr_gap:
        return 40, "CTR_GAP_ONLY", "REVIEW_TITLE_META"
    else:
        return 5, "HEALTHY", "NO_ACTION"

results = [score_row(row, gap) for (_, row), gap in zip(df.iterrows(), df['ctr_gap'])]
df[['score', 'reason_code', 'action']] = pd.DataFrame(results, index=df.index)

queue = df.sort_values('score', ascending=False)[
    ['content_id', 'score', 'reason_code', 'action']
]

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(len(queue), "rows written")
queue.head(20)

30000 rows written


,content_id,score,reason_code,action
29530,content_573248f65db2,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
26897,content_56d7248c0b43,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
6103,content_691fc6b910e6,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
21984,content_02b0d6e30129,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
5653,content_10b9f5f766b4,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
18652,content_0173fb0dc986,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
27701,content_b6e4581523ed,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
8631,content_e2b702f4f92b,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
16163,content_b81e0f46cea9,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW
1361,content_74961b456728,90,STALE_AND_UNDERPERFORMING,REFRESH_NOW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)
for _, r in top20.iterrows():
    print(f"{r['content_id']} | action={r['action']} | reason={r['reason_code']} "
          f"| confidence=? | wrong-if=?")

content_573248f65db2 | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_56d7248c0b43 | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_691fc6b910e6 | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_02b0d6e30129 | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_10b9f5f766b4 | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_0173fb0dc986 | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_b6e4581523ed | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_e2b702f4f92b | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_b81e0f46cea9 | action=REFRESH_NOW | reason=STALE_AND_UNDERPERFORMING | confidence=? | wrong-if=?
content_74961b456728 | action=REFRESH_NOW | reason=STAL

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks: rows where you have low confidence despite a high score
weak = queue[queue['content_id'].isin([...])]  # pick 2-3 from the top20 you're least sure about
print(weak)

# Leakage check: list every column you used in score_row and eyeball each one —
# is it knowable BEFORE you'd act, or does it only exist because the outcome already happened?
used_columns = ['days_since_update', 'ctr', 'avg_position', 'expected_ctr_for_position']
print("Columns used in rule:", used_columns)
print("None of these are post-outcome or future-window — confirm manually against your data dictionary.")

Empty DataFrame
Columns: [content_id, score, reason_code, action]
Index: []
Columns used in rule: ['days_since_update', 'ctr', 'avg_position', 'expected_ctr_for_position']
None of these are post-outcome or future-window — confirm manually against your data dictionary.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.